# CNN-LSTM 深度学习选股策略 (TensorFlow)

## 模型架构

```
输入特征 → Conv1D (局部特征提取) → LSTM (时序建模) → Dense → 输出预测
```

## 特点
- **CNN**: 提取特征的局部模式
- **LSTM**: 捕捉时序依赖
- **TensorFlow/Keras**: 更易用的深度学习框架

In [ ]:
import sys
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

# TensorFlow
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers, Model
    from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
    HAS_TF = True
    print(f"✅ TensorFlow 版本: {tf.__version__}")
except ImportError:
    HAS_TF = False
    raise ImportError("请安装 TensorFlow: pip install tensorflow")

from utils.factor_calculator import TechnicalFactorCalculator
from utils.stock_data_loader import StockDataLoader

# 参数
SEQ_LENGTH = 20      # 序列长度（过去20天）
PRED_HORIZON = 5     # 预测未来5天
TOP_K = 10           # 选股数量

plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print("✅ 环境配置完成")

## 1. 数据加载与序列构建

In [ ]:
# 加载数据
loader = StockDataLoader()
codes = loader.select_sample_codes('hs300', n=50, random_select=True)

# 从CSV加载
df = loader.load_from_csv(
    './data/a_stock_history_price_20260223.csv',
    select_codes=codes
)

# 计算特征
calc = TechnicalFactorCalculator()
feature_list = []

for code in df['code'].unique():
    stock = df[df['code'] == code].copy()
    stock = stock.sort_values('date')
    stock = calc.moving_average(stock)
    stock = calc.rsi(stock)
    stock = calc.macd(stock)
    stock = calc.volatility_factors(stock)
    stock['target'] = stock['close'].pct_change(PRED_HORIZON).shift(-PRED_HORIZON)
    feature_list.append(stock)

df_all = pd.concat(feature_list)
df_all = df_all.dropna()
print(f"\n✅ 数据准备完成: {len(df_all):,} 条")

## 2. 构建时序数据集

In [ ]:
def create_sequences(data_df, feature_cols, seq_length=20):
    """
    构建时序序列数据
    X: (samples, seq_length, features)
    y: (samples,)
    """
    X, y, dates, codes = [], [], [], []
    
    for code in data_df['code'].unique():
        stock = data_df[data_df['code'] == code].sort_values('date')
        stock_values = stock[feature_cols].values
        targets = stock['target'].values
        
        for i in range(len(stock) - seq_length):
            X.append(stock_values[i:i+seq_length])
            y.append(targets[i+seq_length-1])
            dates.append(stock.iloc[i+seq_length-1]['date'])
            codes.append(code)
    
    return np.array(X), np.array(y), dates, codes

# 特征列
feature_cols = [c for c in df_all.columns if any(x in c for x in ['ma_', 'rsi_', 'macd_', 'volatility_'])]
print(f"特征数: {len(feature_cols)}")

# 构建序列
X_all, y_all, dates_all, codes_all = create_sequences(df_all, feature_cols, SEQ_LENGTH)
print(f"序列数据: X{X_all.shape}, y{y_all.shape}")

## 3. 时间序列划分

In [ ]:
# 按日期划分
dates_unique = sorted(list(set(dates_all)))
n_dates = len(dates_unique)

train_end = int(n_dates * 0.6)
val_end = int(n_dates * 0.8)

train_dates = set(dates_unique[:train_end])
val_dates = set(dates_unique[train_end:val_end])
test_dates = set(dates_unique[val_end:])

# 划分数据
train_mask = [d in train_dates for d in dates_all]
val_mask = [d in val_dates for d in dates_all]
test_mask = [d in test_dates for d in dates_all]

X_train, y_train = X_all[train_mask], y_all[train_mask]
X_val, y_val = X_all[val_mask], y_all[val_mask]
X_test, y_test = X_all[test_mask], y_all[test_mask]

# 标准化
scaler = StandardScaler()
X_train_reshaped = X_train.reshape(-1, X_train.shape[-1])
X_train = scaler.fit_transform(X_train_reshaped).reshape(X_train.shape)
X_val = scaler.transform(X_val.reshape(-1, X_val.shape[-1])).reshape(X_val.shape)
X_test = scaler.transform(X_test.reshape(-1, X_test.shape[-1])).reshape(X_test.shape)

print(f"训练: {len(X_train)}, 验证: {len(X_val)}, 测试: {len(X_test)}")

## 4. 构建 CNN-LSTM 模型

In [ ]:
def build_cnn_lstm_model(input_shape):
    """
    CNN-LSTM 混合模型
    """
    inputs = layers.Input(shape=input_shape)
    
    # CNN 层 - 提取局部特征
    x = layers.Conv1D(64, kernel_size=3, activation='relu', padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(pool_size=2)(x)
    x = layers.Dropout(0.3)(x)
    
    x = layers.Conv1D(32, kernel_size=3, activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(pool_size=2)(x)
    x = layers.Dropout(0.3)(x)
    
    # LSTM 层 - 时序建模
    x = layers.LSTM(64, return_sequences=True)(x)
    x = layers.Dropout(0.3)(x)
    x = layers.LSTM(32)(x)
    x = layers.Dropout(0.3)(x)
    
    # Dense 层
    x = layers.Dense(32, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    
    outputs = layers.Dense(1)(x)
    
    model = Model(inputs, outputs)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='mse',
        metrics=['mae']
    )
    return model

# 创建模型
model = build_cnn_lstm_model((SEQ_LENGTH, len(feature_cols)))
model.summary()

## 5. 训练模型

In [ ]:
# 回调函数
callbacks = [
    EarlyStopping(patience=10, restore_best_weights=True),
    ReduceLROnPlateau(factor=0.5, patience=5)
]

# 训练
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=256,
    callbacks=callbacks,
    verbose=1
)

# 绘制学习曲线
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Val')
plt.title('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['mae'], label='Train')
plt.plot(history.history['val_mae'], label='Val')
plt.title('MAE')
plt.legend()
plt.savefig('./data/cnn_lstm_learning_curves.png')
plt.show()

## 6. 预测与回测

In [ ]:
# 预测
y_pred = model.predict(X_test).flatten()

# 评估
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
print(f"RMSE: {rmse:.4f}, R²: {r2:.4f}")

# 构建回测数据
test_df = pd.DataFrame({
    'date': [d for d, m in zip(dates_all, test_mask) if m],
    'code': [c for c, m in zip(codes_all, test_mask) if m],
    'pred': y_pred,
    'actual': y_test
})

# 每日选Top K
daily_returns = []
for date, group in test_df.groupby('date'):
    top = group.nlargest(TOP_K, 'pred')
    avg_return = top['actual'].mean()
    daily_returns.append({'date': date, 'return': avg_return})

returns_df = pd.DataFrame(daily_returns)
returns_df['cum'] = (1 + returns_df['return']).cumprod() - 1

# 绘制回测曲线
plt.figure(figsize=(12, 5))
plt.plot(returns_df['date'], returns_df['cum'] * 100)
plt.axhline(0, color='red', linestyle='--')
plt.title(f'CNN-LSTM Strategy (Top {TOP_K})')
plt.ylabel('Cumulative Return (%)')
plt.savefig('./data/cnn_lstm_backtest.png')
plt.show()

print(f"\n总收益: {returns_df['cum'].iloc[-1]*100:.2f}%")